# Visual Sort — transfer learning on a 3-class CIFAR slice

Warehouse story: sort stills into **vehicle / animal / other-object**.
Stand-in data: CIFAR-10 classes automobile, cat, ship.

Protocol: ImageNet ResNet18 freeze backbone, train a 3-way head,
checkpoint on val loss, test report, one-image inference.

Colab: Runtime GPU. Do not resize back to 32x32 to go faster.


In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.datasets import CIFAR10
from torchvision.models import resnet18, ResNet18_Weights

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__)
print('device:', device)


## Task 0 — inspect, do not train


In [ ]:
import os, tarfile, urllib.request
ROOT = './data'
TGZ = os.path.join(ROOT, 'cifar-10-python.tar.gz')
READY = os.path.join(ROOT, 'cifar-10-batches-py', 'data_batch_1')
MIRRORS = [
    'https://ossci-datasets.s3.amazonaws.com/cifar/cifar-10-python.tar.gz',
    'https://data.brainchip.com/dataset-mirror/cifar10/cifar-10-python.tar.gz',
]
os.makedirs(ROOT, exist_ok=True)
if not os.path.exists(READY):
    last_err = None
    for url in MIRRORS:
        try:
            print('downloading', url)
            urllib.request.urlretrieve(url, TGZ)
            last_err = None
            break
        except Exception as e:
            print('mirror failed:', e)
            last_err = e
    if last_err and not os.path.exists(TGZ):
        raise last_err
    with tarfile.open(TGZ, 'r:gz') as tar:
        tar.extractall(ROOT)
    print('extracted')
else:
    print('cifar already on disk')


In [ ]:
train_raw = CIFAR10(root='./data', train=True, download=False)
test_raw = CIFAR10(root='./data', train=False, download=False)
print('official train / test:', len(train_raw), len(test_raw))
print('all 10 classes:', train_raw.classes)
image, label = train_raw[0]
print('one item:', type(image), image.size, label, train_raw.classes[label])


In [ ]:
KEEP = {1, 3, 8}
NAME = {1: 'automobile', 3: 'cat', 8: 'ship'}

def kept_indices(dataset):
    return [i for i, y in enumerate(dataset.targets) if y in KEEP]

def count_kept(dataset):
    counts = {1: 0, 3: 0, 8: 0}
    for y in dataset.targets:
        if y in counts:
            counts[y] += 1
    return counts

train_idx = kept_indices(train_raw)
test_idx = kept_indices(test_raw)
print('Train kept:', len(train_idx))
for k, n in count_kept(train_raw).items():
    print(f'  {NAME[k]:<12} (id {k}): {n}')
print('Test kept:', len(test_idx))
for k, n in count_kept(test_raw).items():
    print(f'  {NAME[k]:<12} (id {k}): {n}')


In [ ]:
samples, need = [], {1: 3, 3: 3, 8: 2}
for i in train_idx:
    image, label = train_raw[i]
    if need.get(label, 0) > 0:
        samples.append((image, label))
        need[label] -= 1
    if sum(need.values()) == 0:
        break
fig, axes = plt.subplots(2, 4, figsize=(8, 4))
for ax, (image, label) in zip(axes.flat, samples):
    ax.imshow(image)
    ax.set_title(f'{NAME[label]}  (id {label})')
    ax.axis('off')
plt.tight_layout()
plt.show()


## Task 1 — pipeline


In [ ]:
CLASS_MAP = {1: 0, 3: 1, 8: 2}
CLASS_NAMES = ['vehicle', 'animal', 'other-object']

class ThreeClassCIFAR(Dataset):
    def __init__(self, base, indices, transform=None):
        self.base = base
        self.indices = indices
        self.transform = transform
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        image, label = self.base[self.indices[i]]
        label = CLASS_MAP[label]
        if self.transform is not None:
            image = self.transform(image)
        return image, label

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

rng = random.Random(42)
shuffled = train_idx.copy()
rng.shuffle(shuffled)
n_val = int(0.2 * len(shuffled))
val_ids, fit_ids = shuffled[:n_val], shuffled[n_val:]
train_set = ThreeClassCIFAR(train_raw, fit_ids, transform=train_tf)
val_set = ThreeClassCIFAR(train_raw, val_ids, transform=eval_tf)
test_set = ThreeClassCIFAR(test_raw, test_idx, transform=eval_tf)
print('train / val / test:', len(train_set), len(val_set), len(test_set))

BATCH = 64 if device.type == 'cuda' else 32
train_loader = DataLoader(train_set, batch_size=BATCH, shuffle=True, num_workers=0)
val_loader = DataLoader(val_set, batch_size=BATCH, shuffle=False, num_workers=0)
test_loader = DataLoader(test_set, batch_size=BATCH, shuffle=False, num_workers=0)
xb, yb = next(iter(train_loader))
print('batch images:', tuple(xb.shape))
print('batch labels:', tuple(yb.shape), yb[:8].tolist())


## Task 2 — model factory


In [ ]:
def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def make_model(num_classes=3, freeze_backbone=True):
    model = resnet18(weights=ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    if freeze_backbone:
        for name, param in model.named_parameters():
            param.requires_grad = name.startswith('fc')
    return model

frozen = make_model(freeze_backbone=True)
open_net = make_model(freeze_backbone=False)
print('trainable, frozen backbone:', count_trainable(frozen))
print('trainable, full network:   ', count_trainable(open_net))
model = frozen.to(device)
print('model device:', next(model.parameters()).device)


## Task 3 — train the feature extractor


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam((p for p in model.parameters() if p.requires_grad), lr=1e-3)
EPOCHS = 5
best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

def run_epoch(loader, train):
    model.train() if train else model.eval()
    total_loss = correct = total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        if train:
            optimizer.zero_grad()
        with torch.set_grad_enabled(train):
            logits = model(images)
            loss = criterion(logits, labels)
            if train:
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader, train=False)
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(va_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(va_acc)
    print(f'epoch {epoch}/{EPOCHS}  train {tr_loss:.4f}/{tr_acc:.3f}  val {va_loss:.4f}/{va_acc:.3f}')
    if va_loss < best_val_loss:
        best_val_loss = va_loss
        torch.save(model.state_dict(), 'best.pt')
        print('  saved best.pt')
    else:
        print('  kept previous best.pt')


In [ ]:
epochs = range(1, len(history['train_loss']) + 1)
plt.figure(figsize=(6, 4))
plt.plot(epochs, history['train_loss'], label='train loss')
plt.plot(epochs, history['val_loss'], label='val loss')
plt.xlabel('epoch'); plt.ylabel('loss'); plt.title('frozen ResNet18 head'); plt.legend(); plt.show()
print('best val loss:', best_val_loss)


## Task 4 — test like an adult


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
model.load_state_dict(torch.load('best.pt', map_location=device))
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for images, labels in test_loader:
        logits = model(images.to(device))
        y_true.extend(labels.tolist())
        y_pred.extend(logits.argmax(1).cpu().tolist())
y_true = np.array(y_true); y_pred = np.array(y_pred)
print(f'test accuracy: {(y_true == y_pred).mean():.3f}\n')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3))
cm = confusion_matrix(y_true, y_pred)
print(cm)
fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(3), CLASS_NAMES, rotation=20)
ax.set_yticks(range(3), CLASS_NAMES)
ax.set_xlabel('predicted'); ax.set_ylabel('true')
for i in range(3):
    for j in range(3):
        ax.text(j, i, int(cm[i, j]), ha='center', va='center')
fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()


## Task 6 — one image, teammate style


In [ ]:
def denorm(t):
    x = t.detach().cpu().clone()
    for c, m, s in zip(range(3), IMAGENET_MEAN, IMAGENET_STD):
        x[c] = x[c] * s + m
    return x.clamp(0, 1).permute(1, 2, 0).numpy()

model.eval()
image, true_id = test_set[0]
with torch.no_grad():
    logits = model(image.unsqueeze(0).to(device))
    probs = torch.softmax(logits, dim=1).squeeze(0).cpu()
pred_id = int(probs.argmax())
print('true:     ', CLASS_NAMES[true_id])
print('predicted:', CLASS_NAMES[pred_id])
for name, p in zip(CLASS_NAMES, probs.tolist()):
    print(f'  {name:<14} {p:.4f}')
plt.figure(figsize=(2.4, 2.4))
plt.imshow(denorm(image))
plt.title(f'pred {CLASS_NAMES[pred_id]}  true {CLASS_NAMES[true_id]}')
plt.axis('off'); plt.show()


## Task 5 optional

Fine-tune last block from best.pt if you want Question G from a second curve.
Unfreeze layer4 + fc, Adam 1e-4, 3 to 5 epochs.
Read notes.md and TECHNIQUES.md next.
